# Часть 4. Нормализация данных перед анализом


#### Выполнили Ефимов Алексей, Гиллер Александр


Подведем краткий итог предшествовавших этапов, посвященных поиску / подготовке данных:


- В датасете содержалось много дубликатов ≈ 500 строк — они были удалены. Названия колонок были записаны в формате <i>snake_case</i>.
- Были подключены сторонние датасеты с большим числом строк. Затем последовало обогащение начального датасета новыми признаками: `budget, genres, revenue, runtime, vote_average_imdb, vote_count_imdb, popularity_*, content_type`. Некоторые колонки начального датасета были переименованы. Преимуществом данного этапа является высокая уверенность в данных, поскольку датасеты в первую очередь объединялись по точному совпадению `imdb_id`, далее - по названию проекта и году выхода.

- После предыдущего этапа осталось некоторое число пропусков, поскольку данных о некоторых проектах в сторонних датасетах не содержалось / они были неполными / их тяжело получить (например, выручка и бюджет). Наша команда решила обратиться к парсингу данных со сторонних агрегаторов: IMDB, Box Office Mojo, TMDB API и The Numbers. Большинство недостающих данных были собраны. **Мы принимаем во внимание, что большое количество пропусков по признакам `budget` и `revenue` значительно ограничат точность расчетов финансовых показателей. Тем не менее наша команда принимает это допущение в рамках анализа**, поскольку оно не мешает выполнить поставленную задачу — предоставить ответ на вопрос: "Какой фильм / сериал стоит снять студии".


Перейдем к рассмотрению упомянутых допущений.

- Во время парсинга наша команда полагалась на внутреннюю поисковую систему агрегаторов (особенно явно это прослеживается в случае с IMDB). Внутренний поиск агрегаторов оказался удобен и точен, однако он показывал самые популярные результаты. Например, в случае с сериалом он показывал данные за все сезоны, хотя в начальном датасете содержалась ссылка на конкретный сезон / некоторые данные найти было невозможно. С другой стороны, самая актуальная информация зачастую как раз и представлена на подобных агреггированных страницах. Зрители склонны редко оценивать все сезоны сериалов. Например, в случае с «Prison Break» на странице с 3 сезоном всего 1000 отзывов, тогда как на титульной странице около 700 тыс. отзывов. Наша команда считает, что именно вторая цифра отражает реальную популярность / ценность медиапродукта и является более полезной в рамках поставленной задачи.

- Все собранные данные современны и могут не отражать действительность в момент появления начального датасета. Например, знаменитый сериал «Breaking Bad» закончил сниматься в начале 2010-х годов, а данные по нему были собраны  нашей командой в 2026 году — оценка могла повыситься / понизиться, число оценивших сериал выросло. Тем не менее это допущение не мешает выполнить поставленную задачу, поскольку в рамках анализа для нашей команды имеют большую ценность не абсолютные, а относительные показатели: лучше / хуже, больше / меньше. Проект, вышедший 10 лет назад и не запомнившийся зрителям, получит больше голосов, его оценка может измениться, но это произойдет и с остальными проектами — итоговое распределение вряд ли сильно исказиться: популярные и значимые медиапродукты останутся таковыми, аутсайдеры внезапно не завоюют популярность.



В рамках данного этапа будет проведена нормализация собранных данных перед анализом и приведение их к единому виду


## Импорт библиотек


In [433]:
import pandas as pd
import numpy as np

## Шаг 1. Подключение датасета


In [434]:
df_result = pd.read_csv("netflix_parsed_3.csv")

df_result.isna().sum()

title                     0
rating                    0
release_year              0
title_lower               0
imdb_id                   0
vote_average_imdb        24
vote_count_imdb          24
budget                  350
revenue                 334
runtime                  24
genres                    2
popularity_tmdb          36
content_type              0
popularity_imdb_rank    273
dtype: int64

## Шаг 2. Анализ результатов

Во время обогащения датасета парсинга мы пытались найти данные для следуюших признаков:

<ul>
    <li><code>budget</code></li>
    <li><code>revenue</code></li>
    <li><code>runtime</code></li>
    <li><code>genres</code></li>
    <li><code>vote_count_imdb</code></li>
    <li><code>vote_average_imdb</code></li>
    <li><code>popularity_tmdb</code></li>
    <li><code>popularity_imdb_rank</code></li>
</ul>


## Шаг 3.  Первичная обработка итогового датасета `df_result`

Во-первых, удалим лишние (вспомогательные) признаки:

- `title_lower` - был нужен только для обогащение датасета на втором шаге
- `imdb_id` - был нужен только для мерджа и парсинга


In [435]:
df_result = df_result.drop(
    columns=[
        "title_lower",
        "imdb_id",
    ]
)

df_result.head(3)

,title,rating,release_year,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,popularity_tmdb,content_type,popularity_imdb_rank
0,White Chicks,PG-13,2004,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",54.8510,movie,316.0
1,Lucky Number Slevin,R,2006,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",21.6070,movie,2345.0
2,Prison Break,TV-14,2008,8.3,670454.0,NaN,NaN,44.0,"Action, Crime, Drama",93.8498,tvSeries,171.0


## Шаг 4. Нормализация признаков


### 4.1 Нормализация `popularity_*` через единый признак `popularity_score`

В ходе парсинга мы получили два источника оценки популярности:

- `popularity_tmdb` - **TMDB popularity**, непрерывная оценка, чем выше, тем популярнее
- `popularity_imdb_rank` - **IMDb popularity**, ранг, чем ниже, тем популярнее


Приведем оба источника к единой шкале перцентилей `[0 – 100]`, где **100 = самый популярный**:

- TMDB: прямой перцентиль (больше значение -> выше перцентиль)
- IMDB rank: инвертированный перцентиль (меньший ранг -> выше перцентиль)

Итоговый `popularity_score` берёт значение TMDB там, где оно есть, иначе — нормализованный IMDB rank.

In [436]:
df_result["popularity_tmdb_pct"] = df_result["popularity_tmdb"].rank(pct=True) * 100
df_result["popularity_imdb_pct"] = (
    1 - df_result["popularity_imdb_rank"].rank(pct=True)
) * 100

df_result["popularity_score"] = df_result["popularity_tmdb_pct"].fillna(
    df_result["popularity_imdb_pct"]
)

df_result = df_result.drop(columns=["popularity_tmdb_pct", "popularity_imdb_pct"])

print(f"Заполнено: {df_result["popularity_score"].notna().sum()}")
print(f"Пропусков: {df_result['popularity_score'].isna().sum()}")
df_result[["popularity_tmdb", "popularity_imdb_rank", "popularity_score"]].describe()

Заполнено: 465
Пропусков: 34


,popularity_tmdb,popularity_imdb_rank,popularity_score
count,463.000000,226.000000,465.000000
mean,20.850303,1791.172566,49.919117
std,30.043324,1443.959323,28.980322
min,0.014300,21.000000,0.215983
25%,3.750250,457.750000,24.730022
50%,11.980000,1586.500000,49.892009
75%,24.591650,2861.250000,74.946004
max,209.606700,4975.000000,100.000000


In [437]:
df_result = df_result.drop(
    columns=[
        "popularity_tmdb",
        "popularity_imdb_rank",
    ]
)


df_result.head(3)

,title,rating,release_year,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,content_type,popularity_score
0,White Chicks,PG-13,2004,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",movie,91.792657
1,Lucky Number Slevin,R,2006,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",movie,71.922246
2,Prison Break,TV-14,2008,8.3,670454.0,NaN,NaN,44.0,"Action, Crime, Drama",tvSeries,96.112311


### 4.2 Нормализуем признак `rating`:


In [438]:
df_result["rating"].unique()

<StringArray>
[   'PG-13',        'R',    'TV-14',    'TV-PG',    'TV-MA',     'TV-Y',
       'NR',       'UR',       'PG',     'TV-G',        'G',    'TV-Y7',
 'TV-Y7-FV']
Length: 13, dtype: str

После изучения каждого значения было решено разбить начальные 13 значений на 3 категории: kids / teens / adults — дети / подростки / взрослые


In [439]:
rating_map = {
    "G": "kids",
    "TV-G": "kids",
    "TV-Y": "kids",
    "TV-Y7": "kids",
    "TV-Y7-FV": "kids",
    "PG": "teens",
    "TV-PG": "teens",
    "PG-13": "teens",
    "TV-14": "teens",
    "R": "adults",
    "TV-MA": "adults",
    "NR": "adults",
    "UR": "adults",
}

df_result["rating"] = df_result["rating"].apply(lambda x: rating_map[x])
df_result["rating"].value_counts()

rating
teens     227
kids      166
adults    106
Name: count, dtype: int64

Замена прошла успешно. По каждой группе есть данные


### 4.3 Консолидация `genres`

Для признака `genres` применим метод Мультиметки для консолидации жанров. Такой вариант пригодится нам в последующем анализе.

- Выделим подгруппы жанров,
- Добавим соответствующие бинарные столбцы.

Мультиметку реализуем с помощью бинарных колонок (one-hot encoding). Каждый фильм может иметь несколько жанров из набора `GENRES_CONSOLIDATED`.


Посмотрим на распределение данных внутря колонки `genres`:


In [440]:
df_result["genres"].apply(
    lambda x: str(x).split(", ") if pd.notna(x) else []
).explode().value_counts()

genres
Comedy             225
Animation          186
Drama              174
Family             147
Adventure          139
Action              82
Romance             57
Crime               56
Fantasy             47
Mystery             37
Thriller            25
Horror              18
Science Fiction     15
Short               15
Documentary         15
Music               12
Sci-Fi               7
Biography            5
TV Movie             5
Talk-Show            4
History              4
Musical              3
War                  3
Reality-TV           3
Game-Show            3
News                 2
Sport                2
Adult                1
Western              1
Name: count, dtype: int64

In [441]:
unique_genres = set()
for genres_str in df_result["genres"]:
    unique_genres.update(
        g.strip()
        for g in str(genres_str).split(", ")
        if pd.notna(genres_str) and g.strip()
    )
print(list(unique_genres))
print(len(list(unique_genres)))

['Documentary', 'Drama', 'Animation', 'Short', 'Game-Show', 'TV Movie', 'Adult', 'Thriller', 'News', 'Horror', 'Romance', 'Adventure', 'Biography', 'Family', 'Musical', 'Sci-Fi', 'Western', 'Talk-Show', 'Action', 'Science Fiction', 'War', 'Music', 'Fantasy', 'History', 'Reality-TV', 'Crime', 'Sport', 'Mystery', 'Comedy']
29


In [442]:
def parse_all_genres(genres_str):
    if pd.isna(genres_str) or genres_str == "":
        return []

    return [g.strip() for g in str(genres_str).split(", ")]


GENRES_CONSOLIDATED = [
    "Action",
    "Comedy",
    "Drama",
    "Family",
    "Fantasy",
    "Other",
    "Thriller",
]

genre_mapper = {
    "Action": "Action",
    "Adventure": "Action",
    "Western": "Action",
    "War": "Action",
    "Crime": "Action",
    "Comedy": "Comedy",
    "Drama": "Drama",
    "Romance": "Drama",
    "History": "Drama",
    "Family": "Family",
    "Animation": "Family",
    "Fantasy": "Fantasy",
    "Sci-Fi": "Fantasy",
    "Science Fiction": "Fantasy",
    "Adult": "Other",
    "Documentary": "Other",
    "Biography": "Other",
    "News": "Other",
    "Talk-Show": "Other",
    "Game-Show": "Other",
    "Reality-TV": "Other",
    "Sport": "Other",
    "Music": "Other",
    "Musical": "Other",
    "Short": "Other",
    "TV Movie": "Other",
    "Thriller": "Thriller",
    "Mystery": "Thriller",
    "Horror": "Thriller",
}

df_result["genres_list"] = df_result["genres"].apply(parse_all_genres)
df_result["genres_consolidated"] = df_result["genres_list"].apply(
    lambda genres_list: list(
        set([genre_mapper.get(g, "Other") for g in genres_list if g])
    )
)

for genre in GENRES_CONSOLIDATED:
    df_result[f"is_{genre.lower()}"] = df_result["genres_consolidated"].apply(
        lambda x: 1 if genre in x else 0
    )

print("Итого распределено по жанрам")
for genre in GENRES_CONSOLIDATED:
    count = df_result[f"is_{genre.lower()}"].sum()
    print(f"{genre}: {count}")

Итого распределено по жанрам
Action: 215
Comedy: 225
Drama: 197
Family: 240
Fantasy: 66
Other: 61
Thriller: 69


Удалим вспомогательные колонки


In [443]:
df_result = df_result.drop(
    columns=[
        "genres_list",
        "genres_consolidated",
    ]
)


df_result.head(3)

,title,rating,release_year,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,content_type,popularity_score,is_action,is_comedy,is_drama,is_family,is_fantasy,is_other,is_thriller
0,White Chicks,teens,2004,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",movie,91.792657,1,1,0,0,0,0,0
1,Lucky Number Slevin,adults,2006,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",movie,71.922246,1,0,1,0,0,0,1
2,Prison Break,teens,2008,8.3,670454.0,NaN,NaN,44.0,"Action, Crime, Drama",tvSeries,96.112311,1,0,1,0,0,0,0


## Шаг 5. Удаление последних пропусков


Проверим оставшиеся пропуски:


In [444]:
df_result.isna().sum()

title                  0
rating                 0
release_year           0
vote_average_imdb     24
vote_count_imdb       24
budget               350
revenue              334
runtime               24
genres                 2
content_type           0
popularity_score      34
is_action              0
is_comedy              0
is_drama               0
is_family              0
is_fantasy             0
is_other               0
is_thriller            0
dtype: int64

Для фильмов и сериалов пропуски в бюджете и выручке — это обычно не случайные пропуски.

- у старых фильмов может не быть данных,
- у сериалов часто нет понятной общей выручки и не раскрывается бюджет
- у малобюджетных проектов данные могут не публиковаться

Поэтому принимаем решение оставить `budget` и `revenue` как есть для проведения ограниченного анализа


Заменим пропуски `runtime` и `popularity_score` на медиану с группировкой по `content_type`

In [445]:
df_result["runtime"] = df_result["runtime"].fillna(
    df_result.groupby("content_type")["runtime"].transform("median")
)
df_result["runtime"] = df_result["runtime"].fillna(df_result["runtime"].median())

df_result["popularity_score"] = df_result["popularity_score"].fillna(
    df_result.groupby("content_type")["popularity_score"].transform("median")
)
df_result["popularity_score"] = df_result["popularity_score"].fillna(
    df_result["popularity_score"].median()
)

Пропуски по `genres` заполнить нельзя - для этого не хватает данных. Было принято решение избавиться от проблемных строк. И к тому же их всего 2.


In [446]:
df_result = df_result.dropna(subset=["genres"])

По признакам `vote_average_imdb` и `vote_count_imdb` есть пропуски в 24 строках. При рассмотрении можно заменить, что и по всем остальным признакам тоже много пропусков. Поэтому принято решение удалить эти строки


In [447]:
df_result[df_result["vote_average_imdb"].isna() & df_result["vote_count_imdb"].isna()]

,title,rating,release_year,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,content_type,popularity_score,is_action,is_comedy,is_drama,is_family,is_fantasy,is_other,is_thriller
9,Once Upon a Time,teens,2016,NaN,NaN,NaN,NaN,45.0,"Comedy, Drama, Mystery",tvEpisode,49.352052,0,1,1,0,0,0,1
15,Black Mirror,adults,2016,NaN,NaN,NaN,NaN,45.0,"Animation, Comedy, Short",tvEpisode,49.352052,0,1,0,1,0,1,0
18,Marvel's Luke Cage,adults,2016,NaN,NaN,NaN,NaN,45.0,News,tvEpisode,49.352052,0,0,0,0,0,1,0
22,Scream,teens,2016,NaN,NaN,15000000.0,173046640.0,45.0,"Music, Musical, Talk-Show",tvEpisode,49.352052,0,0,0,0,0,1,0
48,Love,adults,2017,NaN,NaN,20000000.0,NaN,45.0,Comedy,tvEpisode,49.352052,0,1,0,0,0,0,0
73,Chef's Table,adults,2017,NaN,NaN,NaN,NaN,45.0,Adult,tvEpisode,49.352052,0,0,0,0,0,1,0
99,BoJack Horseman,adults,2016,NaN,NaN,NaN,NaN,45.0,"Animation, Comedy, Horror",tvEpisode,49.352052,0,1,0,1,0,0,1
106,Haters Back Off,teens,2016,NaN,NaN,NaN,NaN,45.0,Comedy,tvEpisode,49.352052,0,1,0,0,0,0,0
131,The Great Gilly Hopkins,teens,2016,NaN,NaN,NaN,67054.0,45.0,Documentary,tvEpisode,49.352052,0,0,0,0,0,1,0
133,One Day at a Time,teens,2017,NaN,NaN,NaN,NaN,45.0,History,tvEpisode,49.352052,0,0,1,0,0,0,0


In [448]:
df_result = df_result.dropna(
    subset=["vote_average_imdb", "vote_count_imdb"]
).reset_index(drop=True)
print(df_result.shape)
df_result.isna().sum()

(475, 18)


title                  0
rating                 0
release_year           0
vote_average_imdb      0
vote_count_imdb        0
budget               331
revenue              314
runtime                0
genres                 0
content_type           0
popularity_score       0
is_action              0
is_comedy              0
is_drama               0
is_family              0
is_fantasy             0
is_other               0
is_thriller            0
dtype: int64

## Шаг 6. Проверка выбросов


In [449]:
def check_outliers_std(df: pd.DataFrame, column: str) -> str:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = (df[column] < lower) | (df[column] > upper)

    print(f"Число выбросов {column} — {df[outliers].shape[0]}")

    return outliers

### 6.1 Выбросы `vote_average_imdb`

In [450]:
outliers = check_outliers_std(df_result, "vote_average_imdb")
df_result[outliers][["title", "content_type", "vote_average_imdb"]].sort_values("vote_average_imdb")

Число выбросов vote_average_imdb — 3


,title,content_type,vote_average_imdb
179,Leverage,tvSeries,2.8
228,Agent F.O.X.,movie,3.1
82,Amy Schumer: The Leather Special,tvSpecial,3.2


Выбросы `vote_average_imdb` - это экстремально низкие оценки реальных проектов, а не ошибки парсинга. Оставляем как есть.


### 6.2 Выбросы `budget`, `revenue`

In [451]:
outliers = check_outliers_std(df_result, "budget")
df_result[outliers][["title", "revenue", "budget"]].sort_values("budget")

Число выбросов budget — 9


,title,revenue,budget
309,The BFG,1.952434e+08,140000000.0
280,Kung Fu Panda 3,5.211708e+08,145000000.0
47,Zootopia,1.023784e+09,150000000.0
209,Chicken Little,3.144327e+08,150000000.0
269,Bee Movie,2.875946e+08,150000000.0
284,Alice Through the Looking Glass,2.994570e+08,170000000.0
239,The Jungle Book,9.665506e+08,175000000.0
17,The Flash,2.714675e+08,200000000.0
49,Finding Dory,1.028571e+09,200000000.0


In [452]:
outliers = check_outliers_std(df_result, "revenue")
df_result[outliers][["title", "budget", "revenue"]].sort_values("revenue")

Число выбросов revenue — 12


,title,budget,revenue
156,Home,135000000.0,3.688710e+08
133,Twilight,37000000.0,3.936168e+08
263,Tarzan,130000000.0,4.481918e+08
277,Hotel Transylvania 2,80000000.0,4.732270e+08
280,Kung Fu Panda 3,145000000.0,5.211708e+08
270,Kung Fu Panda,130000000.0,6.317446e+08
241,The Secret Life of Pets,75000000.0,8.754579e+08
239,The Jungle Book,175000000.0,9.665506e+08
219,The Super Mario Bros. Super Show!,110000000.0,1.001101e+09
47,Zootopia,150000000.0,1.023784e+09


### 6.3 Выбросы `runtime`

Посмотрим на конкретные значения выбросов `runtime` - при парсинге могли попасть явно ошибочные значения


In [453]:
outliers = check_outliers_std(df_result, "runtime")
df_result[outliers][["title", "content_type", "runtime"]].sort_values("runtime")

Число выбросов runtime — 4


,title,content_type,runtime
75,Cooper Barrett's Guide to Surviving Life,tvSeries,280.0
118,The Carrie Diaries,tvSeries,585.0
310,The Real Ghostbusters,tvSeries,998.0
68,Better Call Saul,tvSeries,999.0


Заменим ошибочные значения медианой по `content_type`

In [454]:
medians_by_type = df_result.groupby("content_type")["runtime"].transform("median")
df_result.loc[outliers, "runtime"] = medians_by_type[outliers]

outliers = check_outliers_std(df_result, "runtime")


Число выбросов runtime — 0


### 6.4 Выбросы `vote_count_imdb`

Ликвидация выбросов по полю `vote_count_imdb`. Устраним выбросы методом логарифмирования:


In [455]:
outliers = check_outliers_std(df_result, "vote_count_imdb")
df_result[outliers][["title", "vote_average_imdb", "vote_count_imdb"]].sort_values(
    "vote_count_imdb"
)

Число выбросов vote_count_imdb — 59


,title,vote_average_imdb,vote_count_imdb
32,Gossip Girl,7.5,214486.0
280,Kung Fu Panda 3,7.1,217919.0
57,The Life Aquatic with Steve Zissou,7.2,220987.0
389,Naruto,8.7,223977.0
288,Chicken Run,7.1,224847.0
50,Sausage Party,6.1,226779.0
241,The Secret Life of Pets,6.5,235617.0
404,Criminal Minds,8.1,239901.0
421,Marvel's Jessica Jones,7.8,240017.0
265,Lilo & Stitch,7.4,243633.0


In [456]:
df_result["vote_count_imdb_log"] = np.log1p(df_result["vote_count_imdb"])
outliers = check_outliers_std(df_result, "vote_count_imdb_log")

Число выбросов vote_count_imdb_log — 0


## Шаг 7. Переименование

Переименуем признак `rating` в `age_group`


In [457]:
df_result = df_result.rename(columns={"rating": "age_group"})

In [458]:
df_result.head(3)

,title,age_group,release_year,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,content_type,popularity_score,is_action,is_comedy,is_drama,is_family,is_fantasy,is_other,is_thriller,vote_count_imdb_log
0,White Chicks,teens,2004,5.9,193002.0,37000000.0,113086475.0,109.0,"Comedy, Crime",movie,91.792657,1,1,0,0,0,0,0,12.170461
1,Lucky Number Slevin,adults,2006,7.7,338824.0,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",movie,71.922246,1,0,1,0,0,0,1,12.733239
2,Prison Break,teens,2008,8.3,670454.0,NaN,NaN,44.0,"Action, Crime, Drama",tvSeries,96.112311,1,0,1,0,0,0,0,13.415712


In [459]:
df_result.isna().sum()

title                    0
age_group                0
release_year             0
vote_average_imdb        0
vote_count_imdb          0
budget                 331
revenue                314
runtime                  0
genres                   0
content_type             0
popularity_score         0
is_action                0
is_comedy                0
is_drama                 0
is_family                0
is_fantasy               0
is_other                 0
is_thriller              0
vote_count_imdb_log      0
dtype: int64

In [460]:
df_result.shape

(475, 19)

In [461]:
df_result.to_csv("netflix_normalized_4.csv", index=False)